# argentina.elecciones — Pruebas interactivas

Recorrido del módulo `argentina.elecciones`.

Estructura:
- **`arg.elecciones`** (core): limpieza/normalización de variables electorales. Solo stdlib, sin red.
- **`arg.elecciones.api`** (opcional): wrappers para APIs electorales. Imports diferidos de `requests`/`pandas`. Instalar con `pip install "argentina[elecciones]"`.

**No** hace scraping de padrón ni consulta datos personales.

## 1. Setup

In [1]:
import argentina as arg

print(f"argentina v{arg.__version__}")
print(f"categorias mapeadas: {len(arg.elecciones.CATEGORIAS)}")
print(f"tipos de eleccion mapeados: {len(arg.elecciones.TIPOS_ELECCION)}")

argentina v0.0.16
categorias mapeadas: 8
tipos de eleccion mapeados: 7


## 2. limpiar_mesa / limpiar_circuito

Mesas: solo dígitos. Circuitos: alfanumérico mayúsculas (los circuitos pueden tener letra: `12A`, `1B`).

In [2]:
for v in ["Mesa 01234", "01234", 1234, "", "abc", None]:
    print(f"{v!r:18} → {arg.elecciones.limpiar_mesa(v)!r}")

'Mesa 01234'       → '01234'
'01234'            → '01234'
1234               → '1234'
''                 → None
'abc'              → None
None               → None


In [3]:
for v in [" 12-A ", "12a", "circuito 5", "01", None]:
    print(f"{v!r:18} → {arg.elecciones.limpiar_circuito(v)!r}")

' 12-A '           → '12A'
'12a'              → '12A'
'circuito 5'       → 'CIRCUITO5'
'01'               → '01'
None               → None


## 3. normalizar_categoria

Mapea variantes (singular/plural, masculino/femenino) a un label canónico.

In [4]:
for v in ["Presidente", "Presidenta", "diputado", "diputados", "senador", "GOBERNADOR", "intendente", "concejal"]:
    print(f"{v!r:14} → {arg.elecciones.normalizar_categoria(v)!r}")

'Presidente'   → 'Presidente'
'Presidenta'   → 'Presidente'
'diputado'     → 'Diputados'
'diputados'    → 'Diputados'
'senador'      → 'Senadores'
'GOBERNADOR'   → 'Gobernador'
'intendente'   → 'Intendente'
'concejal'     → None


## 4. normalizar_tipo_eleccion

PASO / General / Ballotage.

In [5]:
for v in ["PASO", "primarias", "general", "generales", "Ballotage", "segunda vuelta", "otra"]:
    print(f"{v!r:18} → {arg.elecciones.normalizar_tipo_eleccion(v)!r}")

'PASO'             → 'PASO'
'primarias'        → 'PASO'
'general'          → 'General'
'generales'        → 'General'
'Ballotage'        → 'Ballotage'
'segunda vuelta'   → 'Ballotage'
'otra'             → None


## 5. validar_anio_eleccion

Rango razonable: 1983 (vuelta de la democracia) a 2100. Acepta int o string.

In [6]:
for v in [1983, 2023, "2027", 1900, 1982, 2101, None, "dos mil"]:
    print(f"{v!r:12} → {arg.elecciones.validar_anio_eleccion(v)}")

1983         → True
2023         → True
'2027'       → True
1900         → False
1982         → False
2101         → False
None         → False
'dos mil'    → False


## 6. Tablas expuestas

Los mapas se exponen como dicts mutables — útiles para extender en runtime si hace falta.

In [7]:
print("CATEGORIAS:")
for k, v in arg.elecciones.CATEGORIAS.items():
    print(f"  {k!r:15} → {v!r}")
print()
print("TIPOS_ELECCION:")
for k, v in arg.elecciones.TIPOS_ELECCION.items():
    print(f"  {k!r:18} → {v!r}")

CATEGORIAS:
  'presidente'    → 'Presidente'
  'presidenta'    → 'Presidente'
  'diputado'      → 'Diputados'
  'diputados'     → 'Diputados'
  'senador'       → 'Senadores'
  'senadores'     → 'Senadores'
  'gobernador'    → 'Gobernador'
  'intendente'    → 'Intendente'

TIPOS_ELECCION:
  'paso'             → 'PASO'
  'primaria'         → 'PASO'
  'primarias'        → 'PASO'
  'general'          → 'General'
  'generales'        → 'General'
  'ballotage'        → 'Ballotage'
  'segunda vuelta'   → 'Ballotage'


## 7. Wrapper opcional `arg.elecciones.api`

Helpers genéricos para APIs HTTP. Importar el módulo no necesita `requests`/`pandas` — el chequeo es diferido.

In [8]:
# Ver qué deps del extra están instaladas
arg.elecciones.api.disponible()

{'requests': True, 'pandas': True}

In [9]:
# Función pública: GET genérico que devuelve JSON
help(arg.elecciones.api.obtener_json)

Help on function obtener_json in module argentina.elecciones.api:

obtener_json(url: 'str', params: 'dict | None' = None, timeout: 'int' = 30)
    GET genérico que devuelve JSON. Requiere `requests`.
    
    Pensado como bloque de construcción para wrappers concretos sobre
    endpoints electorales (resultados.gob.ar, escrutinios provinciales, etc.).



Uso típico (descomentar para probar contra una API real):

```python
datos = arg.elecciones.api.obtener_json(
    "https://URL_OFICIAL/resultados.json",
    params={"anio": 2023},
)
```

## 8. Combinando todo

Pipeline típico: filas crudas de un acta electoral, normalizar antes de agregar.

In [10]:
registros = [
    {"anio": 2023, "tipo": "PASO",      "categoria": "Presidente", "mesa": "Mesa 01234", "circuito": "12-A"},
    {"anio": 2023, "tipo": "general",   "categoria": "diputados",  "mesa": 5678,         "circuito": "3b"},
    {"anio": 2023, "tipo": "ballotage", "categoria": "Presidenta", "mesa": "",           "circuito": None},
    {"anio": 1900, "tipo": "otra",      "categoria": "concejal",   "mesa": None,         "circuito": " "},
]

for r in registros:
    print({
        "anio_valido":  arg.elecciones.validar_anio_eleccion(r["anio"]),
        "tipo":         arg.elecciones.normalizar_tipo_eleccion(r["tipo"]),
        "categoria":    arg.elecciones.normalizar_categoria(r["categoria"]),
        "mesa":         arg.elecciones.limpiar_mesa(r["mesa"]),
        "circuito":     arg.elecciones.limpiar_circuito(r["circuito"]),
    })

{'anio_valido': True, 'tipo': 'PASO', 'categoria': 'Presidente', 'mesa': '01234', 'circuito': '12A'}
{'anio_valido': True, 'tipo': 'General', 'categoria': 'Diputados', 'mesa': '5678', 'circuito': '3B'}
{'anio_valido': True, 'tipo': 'Ballotage', 'categoria': 'Presidente', 'mesa': None, 'circuito': None}
{'anio_valido': False, 'tipo': None, 'categoria': None, 'mesa': None, 'circuito': None}


## 9. Tests automáticos

```bash
cd /Users/tobiasyatche/argentina
pytest tests/test_elecciones_core.py tests/test_elecciones_api.py -v
```

## Notas sueltas / TODOs

- Core: solo stdlib. Mapas (`CATEGORIAS`, `TIPOS_ELECCION`) son `dict` plano — extensibles en runtime.
- `validar_anio_eleccion` no chequea que efectivamente hubo una elección ese año (no tiene calendario embebido); solo rango plausible 1983–2100.
- `arg.elecciones.api` queda como esqueleto: tiene `obtener_json` genérico para que se puedan armar wrappers concretos sobre `resultados.gob.ar`, escrutinios provinciales, etc., sin meter dependencia obligatoria a `requests`.
- **Nada de scraping de padrón** ni consulta de DNI: queda explícitamente fuera del scope.